## Increasing-units grand-average PSTH — visualisation
Author: patrick.mccarthy@dpag.ox.ac.uk

Loads the output of `scripts/05_06_26_extract_increasing_units_psth.py`.
For each (session × stimulus) a unit is "increasing" if its trial-averaged peak
firing rate in the classification window exceeds its pre-stimulus baseline.
Only those units contribute to the PSTH for that stimulus.

| Event | Time |
|---|---|
| Stim onset | t = 0 ms |
| Stim offset | t ≈ 250 ms |
| Classification window | 50 – 175 ms |
| Baseline window | −50 – 0 ms |

In [ ]:
import pickle
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.ndimage import gaussian_filter1d

In [ ]:
PKL_PATH = Path('/Users/pmccarthy/Documents/experimental_data/'
                'allen_visual_neuropixels_longwindow_5ms_bins/'
                'increasing_units_grand_avg_psth.pkl')

ALL_UNITS_PKL = Path('/Users/pmccarthy/Documents/designer-waveform/'
                     'data/allen_visual_neuropixels_passive/visp_psths.pkl')

SPIKES_DIR = Path('/Users/pmccarthy/Documents/experimental_data/'
                  'allen_visual_neuropixels_longwindow_5ms_bins/spike_times_v2')

FIG_DIR = Path('/Users/pmccarthy/Documents/designer-waveform/results/increasing_units_vis')
FIG_DIR.mkdir(parents=True, exist_ok=True)

with open(PKL_PATH, 'rb') as f:
    data = pickle.load(f)

meta        = data['meta']
grand_avg   = data['grand_avg']
per_session = data['per_session']

t_ms       = grand_avg['t_ms']
mu         = grand_avg['mean_hz']
sem        = grand_avg['sem_hz']
n_sessions = grand_avg['n_sessions']

T_STIM_ON  = 0.0
T_STIM_OFF = 250.0

CWIN_LO = meta['stim_win_lo_ms']
CWIN_HI = meta['stim_win_hi_ms']
BL_LO   = meta['baseline_lo_ms']
BL_HI   = meta['baseline_hi_ms']
SMOOTH_SIGMA_MS = meta.get('smooth_sigma_ms', 15.0) or 15.0

print(f'Loaded  : {PKL_PATH.name}')
print(f'Sessions: {n_sessions} contributing  /  {meta["n_sessions_total"]} total')
print(f'Bins    : {len(t_ms)} × {meta["bin_size_ms"]:.0f} ms')
print(f'Window  : {meta["T_PRE_ms"]:.0f} – {meta["T_POST_ms"]:.0f} ms')
print(f'Class.  : {CWIN_LO:.0f} – {CWIN_HI:.0f} ms  |  baseline {BL_LO:.0f} – {BL_HI:.0f} ms')
print(f'Unimodal: {meta.get("require_unimodal")}  |  smooth σ = {SMOOTH_SIGMA_MS:.0f} ms')

### Grand-average PSTH — increasing units only

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.fill_between(t_ms, mu - sem, mu + sem, color='steelblue', alpha=0.25)
ax.plot(t_ms, mu, color='steelblue', lw=1.8, label=f'mean ± SEM  (n={n_sessions} sessions)')
ax.axvline(T_STIM_ON,  color='tab:blue',   lw=1.0, ls='--', label='stim onset')
ax.axvline(T_STIM_OFF, color='tab:orange', lw=1.0, ls='--', label='stim offset')
ax.axvspan(CWIN_LO, CWIN_HI, color='steelblue', alpha=0.10,
           label=f'classification window ({CWIN_LO:.0f}–{CWIN_HI:.0f} ms)')
ax.axvspan(BL_LO, BL_HI, color='grey', alpha=0.10, label=f'baseline ({BL_LO:.0f}–{BL_HI:.0f} ms)')
ax.set_xlabel('Time from stim onset (ms)')
ax.set_ylabel('Firing rate (Hz)')
ax.set_title('Grand-average PSTH — increasing units only (passive natural scenes)')
ax.legend(frameon=False, fontsize=8, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / 'grand_avg_increasing_units.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Peak: {mu.max():.2f} Hz at {t_ms[np.argmax(mu)]:.0f} ms')

### Per-session traces — cross-session variability

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for s in per_session:
    ax.plot(t_ms, s['mean_hz'], color='steelblue', lw=0.6, alpha=0.3)
ax.fill_between(t_ms, mu - sem, mu + sem, color='steelblue', alpha=0.25)
ax.plot(t_ms, mu, color='steelblue', lw=2.0, label='cross-session mean ± SEM')
ax.axvline(T_STIM_ON,  color='tab:blue',   lw=1.0, ls='--', label='stim onset')
ax.axvline(T_STIM_OFF, color='tab:orange', lw=1.0, ls='--', label='stim offset')
ax.axvspan(CWIN_LO, CWIN_HI, color='steelblue', alpha=0.08)
ax.set_xlabel('Time from stim onset (ms)')
ax.set_ylabel('Firing rate (Hz)')
ax.set_title(f'Per-session traces (faint) + grand average (bold)  —  n={n_sessions} sessions')
ax.legend(frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / 'per_session_traces.png', dpi=150, bbox_inches='tight')
plt.show()

### Comparison — increasing units vs all-units average

In [ ]:
if ALL_UNITS_PKL is None or not ALL_UNITS_PKL.exists():
    print(f'All-units pkl not found ({ALL_UNITS_PKL}) — skipping.')
else:
    with open(ALL_UNITS_PKL, 'rb') as f:
        _all = pickle.load(f)
    _fkey  = 'filtered' if 'filtered' in _all else 'unfiltered'
    _entry = _all[_fkey]['all']
    t_all  = _all['meta']['time_ms']
    mu_all = _entry['mean_hz']
    se_all = _entry['sem_hz']
    win    = (t_all >= -50) & (t_all <= T_STIM_OFF)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.fill_between(t_all[win], mu_all[win] - se_all[win], mu_all[win] + se_all[win], color='grey', alpha=0.2)
    ax.plot(t_all[win], mu_all[win], color='grey', lw=1.8, label=f'all units ({_fkey})')
    inc_win = (t_ms >= -50) & (t_ms <= T_STIM_OFF)
    ax.fill_between(t_ms[inc_win], mu[inc_win] - sem[inc_win], mu[inc_win] + sem[inc_win], color='steelblue', alpha=0.25)
    ax.plot(t_ms[inc_win], mu[inc_win], color='steelblue', lw=1.8, label='increasing units only')
    ax.axvline(T_STIM_ON,  color='tab:blue',   lw=1.0, ls='--', label='stim onset')
    ax.axvline(T_STIM_OFF, color='tab:orange', lw=1.0, ls='--', label='stim offset')
    ax.axvspan(CWIN_LO, CWIN_HI, color='steelblue', alpha=0.07)
    ax.set_xlabel('Time from stim onset (ms)')
    ax.set_ylabel('Firing rate (Hz)')
    ax.set_title('Increasing units vs all-units grand-average PSTH')
    ax.legend(frameon=False, fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'increasing_vs_all_units.png', dpi=150, bbox_inches='tight')
    plt.show()

### Layer-resolved increasing-units PSTHs

Re-loads the raw spike-time pickles and applies the same peak-vs-baseline
classification within each CCF layer separately.  
**Run the Load + bin cell once before any of the cells below.**

In [ ]:
# ── Load + bin ────────────────────────────────────────────────────────────────
LAYER_ORDER  = ['VISp1', 'VISp2/3', 'VISp2/3a', 'VISp2/3b', 'VISp4', 'VISp5', 'VISp6a', 'VISp6b']
LAYER_COLORS = {
    'VISp1':    '#a6cee3', 'VISp2/3':  '#1f78b4', 'VISp2/3a': '#4393c3',
    'VISp2/3b': '#2166ac', 'VISp4':    '#33a02c', 'VISp5':    '#e31a1c',
    'VISp6a':   '#ff7f00', 'VISp6b':   '#cab2d6',
}

_bin_s    = meta['bin_size_ms'] / 1000.0
_t_pre_s  = meta['T_PRE_ms']   / 1000.0
_t_post_s = meta['T_POST_ms']  / 1000.0
bin_edges = np.arange(-_t_pre_s, _t_post_s + _bin_s, _bin_s)
t_lyr     = (bin_edges[:-1] + bin_edges[1:]) / 2 * 1000

bl_mask        = (t_lyr >= BL_LO)   & (t_lyr <  BL_HI)
cwin_mask      = (t_lyr >= CWIN_LO) & (t_lyr <= CWIN_HI)
stim_full_mask = (t_lyr >= T_STIM_ON) & (t_lyr <= T_STIM_OFF)

pkl_files = sorted(SPIKES_DIR.glob('*_alllayers_spiketimes.pkl'))
print(f'Found {len(pkl_files)} spike-time pkl files — binning...')

session_data = []
layer_traces = defaultdict(list)

for pkl_path in pkl_files:
    with open(pkl_path, 'rb') as _f:
        s = pickle.load(_f)
    spikes = s['spikes']
    layers = np.asarray(s['layer'])
    num_stim, num_trials, num_units = spikes.shape
    if num_units == 0:
        continue

    psth = np.zeros((num_stim, num_units, len(bin_edges) - 1), dtype=np.float32)
    for si in range(num_stim):
        for k in range(num_units):
            counts = np.zeros(len(bin_edges) - 1, dtype=np.float32)
            for ti in range(num_trials):
                st = spikes[si, ti, k]
                if len(st) > 0:
                    counts += np.histogram(st, bins=bin_edges)[0]
            psth[si, k] = counts / (num_trials * _bin_s)

    baseline_mean = psth[:, :, bl_mask].mean(axis=2)
    stim_peak     = psth[:, :, cwin_mask].max(axis=2)
    increasing    = stim_peak > baseline_mean

    sigma_bins  = SMOOTH_SIGMA_MS / (meta['bin_size_ms'])
    psth_smooth = gaussian_filter1d(psth.astype(np.float32), sigma=sigma_bins, axis=2)
    p        = psth_smooth[:, :, stim_full_mask]
    inner    = p[:, :, 1:-1]
    has_dip  = ((inner < p[:, :, :-2]) & (inner < p[:, :, 2:])).any(axis=2)
    increasing_unimodal = increasing & ~has_dip

    n_bins = psth.shape[2]
    unit_psth_selected = np.full((num_units, n_bins), np.nan, dtype=np.float32)
    for k in range(num_units):
        sel = np.where(increasing_unimodal[:, k])[0]
        if len(sel) > 0:
            unit_psth_selected[k] = psth[sel, k, :].mean(axis=0)

    session_data.append({
        'session_id':          s['session_id'],
        'psth':                psth,
        'layers':              layers,
        'increasing':          increasing,
        'increasing_unimodal': increasing_unimodal,
        'unit_psth_selected':  unit_psth_selected,
    })

    for lyr in np.unique(layers):
        lyr_mask    = layers == lyr
        stim_traces = []
        for si in range(num_stim):
            inc = increasing_unimodal[si] & lyr_mask
            if inc.any():
                stim_traces.append(psth[si, inc].mean(axis=0))
        if stim_traces:
            layer_traces[lyr].append(np.mean(stim_traces, axis=0))

layers_present = [l for l in LAYER_ORDER if l in layer_traces] + \
                 [l for l in layer_traces if l not in LAYER_ORDER]
print('Done.  Sessions per layer:')
for lyr in layers_present:
    print(f'  {lyr}: {len(layer_traces[lyr])} sessions')

In [ ]:
# ── Stacked subplots ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(len(layers_present), 1,
                          figsize=(7, 2.2 * len(layers_present)), sharex=True)
if len(layers_present) == 1:
    axes = [axes]
for ax, lyr in zip(axes, layers_present):
    color  = LAYER_COLORS.get(lyr, '#888888')
    traces = np.stack(layer_traces[lyr])
    lmu    = traces.mean(axis=0)
    lsem   = traces.std(axis=0) / np.sqrt(traces.shape[0])
    ax.fill_between(t_lyr, lmu - lsem, lmu + lsem, color=color, alpha=0.25)
    ax.plot(t_lyr, lmu, color=color, lw=1.5)
    ax.axvline(T_STIM_ON,  color='tab:blue',   lw=0.8, ls='--')
    ax.axvline(T_STIM_OFF, color='tab:orange', lw=0.8, ls='--')
    ax.axvspan(CWIN_LO, CWIN_HI, color=color, alpha=0.07)
    ax.set_ylabel('Hz', fontsize=8)
    ax.set_title(f'{lyr}  (n={len(layer_traces[lyr])} sess)', fontsize=9, loc='left', pad=2)
    ax.spines[['top', 'right']].set_visible(False)
axes[-1].set_xlabel('Time from stim onset (ms)')
fig.suptitle('Increasing-units PSTH by layer — cross-session mean ± SEM', y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / 'layer_stacked.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Overlay ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
for lyr in layers_present:
    color  = LAYER_COLORS.get(lyr, '#888888')
    traces = np.stack(layer_traces[lyr])
    lmu    = traces.mean(axis=0)
    lsem   = traces.std(axis=0) / np.sqrt(traces.shape[0])
    ax.plot(t_lyr, lmu, color=color, lw=1.5, label=f'{lyr} (n={len(layer_traces[lyr])})')
    ax.fill_between(t_lyr, lmu - lsem, lmu + lsem, color=color, alpha=0.12)
ax.plot(t_ms, mu, color='k', lw=2.2, zorder=5, label=f'all VISp (n={n_sessions} sess)')
ax.fill_between(t_ms, mu - sem, mu + sem, color='k', alpha=0.12, zorder=4)
ax.axvline(T_STIM_ON,  color='tab:blue',   lw=0.9, ls='--')
ax.axvline(T_STIM_OFF, color='tab:orange', lw=0.9, ls='--')
ax.axvspan(CWIN_LO, CWIN_HI, color='grey', alpha=0.07)
ax.set_xlabel('Time from stim onset (ms)')
ax.set_ylabel('Firing rate (Hz)')
ax.set_title('Increasing-units PSTH — layer overlay + grand average')
ax.legend(frameon=False, fontsize=8, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)
ax.set_xlim(-_t_pre_s * 1000, T_STIM_OFF)
fig.tight_layout()
fig.savefig(FIG_DIR / 'layer_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

### Save layer-filtered target — for optimisation

Set `TARGET_LAYERS` to whichever layers you want to include, then run this cell.
It saves a pkl that the optimisation notebook can load directly.

**Presets:**
```python
TARGET_LAYERS = ['VISp2/3', 'VISp4', 'VISp5', 'VISp6a', 'VISp6b']  # canonical layers only
TARGET_LAYERS = ['VISp2/3']                                           # L2/3 only
TARGET_LAYERS = None                                                   # all layers (uses grand_avg pkl directly)
```

**Loading in the optimisation notebook** — add to the `_path_map` dict:
```python
_path_map['increasing_l23']            = Path('../data/allen_visual_neuropixels_passive/increasing_units_l23_target.pkl')
_path_map['increasing_canonical_layers'] = Path('../data/allen_visual_neuropixels_passive/increasing_units_canonical_layers_target.pkl')
```
Then add a loading branch (replace the existing `_entry = ...` block):
```python
if SOURCE.startswith('increasing_'):
    _ga         = _allen['grand_avg']
    _meta       = _allen['meta']
    t_ms_full   = _meta['t_ms']
    psth_hz     = _ga['mean_hz']
    psth_err_hz = _ga['sem_hz']
    bin_size_ms = float(_meta['bin_size_ms'])
    T_PRE_ms    = float(_meta['T_PRE_ms'])
    T_POST_ms   = float(_meta['T_POST_ms'])
```

In [ ]:
# ── parameters ────────────────────────────────────────────────────────────────
# Choose one of the presets above or define your own list of layer strings.
TARGET_LAYERS = ['VISp2/3', 'VISp4', 'VISp5', 'VISp6a', 'VISp6b']   # ← edit here

# Output directory — copy into the data folder the optimisation nb already reads from
TARGET_OUT_DIR = Path('/Users/pmccarthy/Documents/designer-waveform/'
                      'data/allen_visual_neuropixels_passive')
TARGET_OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── compute layer-filtered grand average ──────────────────────────────────────
assert session_data, 'Run the Load + bin cell first.'

if TARGET_LAYERS is None:
    # use the grand_avg from the pkl as-is (all layers)
    _mu  = mu
    _sem = sem
    _n   = n_sessions
    tag  = 'all_layers'
else:
    # re-average over selected layers only
    sess_traces = []
    lyr_set = set(TARGET_LAYERS)
    for sd in session_data:
        lyr_mask  = np.isin(sd['layers'], list(lyr_set))
        stim_traces = []
        num_stim = sd['psth'].shape[0]
        for si in range(num_stim):
            inc = sd['increasing_unimodal'][si] & lyr_mask
            if inc.any():
                stim_traces.append(sd['psth'][si, inc].mean(axis=0))
        if stim_traces:
            sess_traces.append(np.mean(stim_traces, axis=0))
    if not sess_traces:
        raise RuntimeError(f'No contributing units found for layers {TARGET_LAYERS}')
    arr  = np.stack(sess_traces)
    _mu  = arr.mean(axis=0)
    _sem = arr.std(axis=0) / np.sqrt(arr.shape[0])
    _n   = len(sess_traces)
    # build filename tag from layer list
    _lyr_short = {'VISp2/3': 'l23', 'VISp4': 'l4', 'VISp5': 'l5',
                  'VISp6a': 'l6a', 'VISp6b': 'l6b', 'VISp1': 'l1'}
    tag = '_'.join(_lyr_short.get(l, l.replace('/', '')) for l in TARGET_LAYERS
                   if l in _lyr_short)

# ── save ──────────────────────────────────────────────────────────────────────
out_path = TARGET_OUT_DIR / f'increasing_units_{tag}_target.pkl'
payload  = {
    'meta': {
        **meta,
        't_ms':         t_lyr,
        'target_layers': TARGET_LAYERS,
        'tag':           tag,
    },
    'grand_avg': {
        'mean_hz':   _mu,
        'sem_hz':    _sem,
        'n_sessions': _n,
        't_ms':      t_lyr,
    },
}
with open(out_path, 'wb') as f:
    pickle.dump(payload, f)

# ── quick preview plot ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3))
ax.fill_between(t_lyr, _mu - _sem, _mu + _sem, color='steelblue', alpha=0.25)
ax.plot(t_lyr, _mu, color='steelblue', lw=1.8,
        label=f'{tag}  (n={_n} sessions)')
ax.axvline(T_STIM_ON,  color='tab:blue',   lw=0.9, ls='--')
ax.axvline(T_STIM_OFF, color='tab:orange', lw=0.9, ls='--')
ax.set_xlabel('Time from stim onset (ms)')
ax.set_ylabel('Firing rate (Hz)')
ax.set_title(f'Target: {tag}  →  {out_path.name}')
ax.legend(frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()

print(f'Saved → {out_path}')
print(f'  layers : {TARGET_LAYERS}')
print(f'  sessions: {_n}  |  bins: {len(t_lyr)} × {meta["bin_size_ms"]:.0f} ms')
print(f'  peak   : {_mu.max():.2f} Hz at {t_lyr[np.argmax(_mu)]:.0f} ms')

### Grand population heatmap — all units, all sessions

Each row is one contributing unit's PSTH averaged over its own selected stimuli.
Units are grouped by layer (horizontal bands, colour-coded on the left spine)
and within each layer by session (thin white dividers).
Two variants: sorted by peak time within each layer, and unsorted (probe order).

In [ ]:
rows       = []
row_layer  = []
row_sess   = []

for lyr in layers_present:
    for sd in session_data:
        lyr_mask = sd['layers'] == lyr
        lyr_idxs = np.where(lyr_mask)[0]
        for k in lyr_idxs:
            trace = sd['unit_psth_selected'][k]
            if not np.isnan(trace[0]):
                rows.append(trace)
                row_layer.append(lyr)
                row_sess.append(sd['session_id'])

mat        = np.stack(rows)
row_layer  = np.array(row_layer)
row_sess   = np.array(row_sess)
n_rows     = mat.shape[0]

layer_bounds  = {}
session_lines = []
pos = 0
for lyr in layers_present:
    mask         = row_layer == lyr
    lyr_start    = pos
    sess_ids_lyr = row_sess[mask]
    unique_sess  = list(dict.fromkeys(sess_ids_lyr))
    for sid in unique_sess[:-1]:
        boundary = pos + (sess_ids_lyr == sid).sum()
        session_lines.append(boundary)
        pos = boundary
    pos += (sess_ids_lyr == unique_sess[-1]).sum()
    layer_bounds[lyr] = (lyr_start, pos)

print(f'Grand matrix: {n_rows} units × {mat.shape[1]} bins')
for lyr in layers_present:
    s, e = layer_bounds[lyr]
    print(f'  {lyr}: {e-s} units')

In [ ]:
# ── Heatmap 1: layer → session order ─────────────────────────────────────────
stim_win = (t_lyr >= T_STIM_ON) & (t_lyr <= T_STIM_OFF)
t_plot   = t_lyr
fig_h    = max(6, n_rows * 0.04 + 2)
vmax     = np.nanpercentile(mat, 98)

fig, (ax_map, ax_cb) = plt.subplots(1, 2, figsize=(9, fig_h),
                                     gridspec_kw={'width_ratios': [0.03, 1]})
layer_cmap = np.zeros((n_rows, 1, 4))
for lyr in layers_present:
    s, e = layer_bounds[lyr]
    layer_cmap[s:e, 0, :] = mpatches.Patch(color=LAYER_COLORS.get(lyr, '#888')).get_facecolor()
ax_map.imshow(layer_cmap, aspect='auto', origin='upper', extent=[0,1,n_rows,0], interpolation='none')
ax_map.set_xticks([])
ax_map.set_ylabel('Unit (grouped by layer, then session)')
for lyr in layers_present:
    s, e = layer_bounds[lyr]
    ax_map.text(-0.5, (s+e)/2, lyr, va='center', ha='right', fontsize=6,
                color=LAYER_COLORS.get(lyr,'#888'), transform=ax_map.get_yaxis_transform(), clip_on=False)

im = ax_cb.imshow(mat, aspect='auto', origin='upper', extent=[t_plot[0],t_plot[-1],n_rows,0],
                  cmap='inferno', vmin=0, vmax=vmax, interpolation='nearest')
for lyr in layers_present:
    s, e = layer_bounds[lyr]
    if s > 0: ax_cb.axhline(s, color='white', lw=1.5, zorder=3)
for y in session_lines:
    ax_cb.axhline(y, color='white', lw=0.4, alpha=0.5, zorder=3)
ax_cb.axvline(T_STIM_ON,  color='deepskyblue', lw=1.0, ls='--')
ax_cb.axvline(T_STIM_OFF, color='orange',      lw=1.0, ls='--')
ax_cb.axvspan(CWIN_LO, CWIN_HI, color='white', alpha=0.06)
ax_cb.set_xlabel('Time from stim onset (ms)')
ax_cb.set_yticks([])
fig.colorbar(im, ax=ax_cb, shrink=0.4, pad=0.01, label='Hz')
fig.suptitle(f'All contributing units — {n_rows} units, {len(session_data)} sessions\n'
             f'thick white = layer  |  thin white = session', fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / 'grand_population_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Heatmap 2: peak-time sorted within each layer ─────────────────────────────
sort_order = []
for lyr in layers_present:
    s, e      = layer_bounds[lyr]
    peak_bins = np.argmax(mat[s:e, :][:, stim_win], axis=1)
    sort_order.extend(np.arange(s, e)[np.argsort(peak_bins)].tolist())
sort_order = np.array(sort_order)
mat_sorted = mat[sort_order]

fig, (ax_map2, ax_cb2) = plt.subplots(1, 2, figsize=(9, fig_h),
                                       gridspec_kw={'width_ratios': [0.03, 1]})
ax_map2.imshow(layer_cmap[sort_order], aspect='auto', origin='upper',
               extent=[0,1,n_rows,0], interpolation='none')
ax_map2.set_xticks([])
ax_map2.set_ylabel('Unit (sorted by peak time within layer)')
for lyr in layers_present:
    s, e = layer_bounds[lyr]
    ax_map2.text(-0.5, (s+e)/2, lyr, va='center', ha='right', fontsize=6,
                 color=LAYER_COLORS.get(lyr,'#888'), transform=ax_map2.get_yaxis_transform(), clip_on=False)

im2 = ax_cb2.imshow(mat_sorted, aspect='auto', origin='upper',
                    extent=[t_plot[0],t_plot[-1],n_rows,0],
                    cmap='inferno', vmin=0, vmax=vmax, interpolation='nearest')
for lyr in layers_present:
    s, e = layer_bounds[lyr]
    if s > 0: ax_cb2.axhline(s, color='white', lw=1.5, zorder=3)
ax_cb2.axvline(T_STIM_ON,  color='deepskyblue', lw=1.0, ls='--')
ax_cb2.axvline(T_STIM_OFF, color='orange',      lw=1.0, ls='--')
ax_cb2.axvspan(CWIN_LO, CWIN_HI, color='white', alpha=0.06)
ax_cb2.set_xlabel('Time from stim onset (ms)')
ax_cb2.set_yticks([])
fig.colorbar(im2, ax=ax_cb2, shrink=0.4, pad=0.01, label='Hz')
fig.suptitle('All contributing units — sorted by peak time within layer', fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / 'grand_population_heatmap_peaksorted.png', dpi=150, bbox_inches='tight')
plt.show()

### Per-unit diagnostic — individual trial-averaged PSTHs

In [ ]:
SESSION_IDX  = 0
TARGET_LAYER = 'VISp2/3'

sd     = session_data[SESSION_IDX]
psth   = sd['psth']
layers = sd['layers']
inc    = sd['increasing']
inc_u  = sd['increasing_unimodal']

lyr_mask = layers == TARGET_LAYER
if not lyr_mask.any():
    print(f'{TARGET_LAYER!r} not found. Available: {np.unique(layers).tolist()}')
else:
    lyr_idxs = np.where(lyr_mask)[0]
    n_units  = len(lyr_idxs)

    psth_selected = np.full((n_units, psth.shape[2]), np.nan)
    psth_inc_only = np.full((n_units, psth.shape[2]), np.nan)
    n_stim_sel    = np.zeros(n_units, dtype=int)

    for ki, k in enumerate(lyr_idxs):
        sel_stim = np.where(inc_u[:, k])[0]
        if len(sel_stim) > 0:
            psth_selected[ki] = psth[sel_stim, k, :].mean(axis=0)
            n_stim_sel[ki]    = len(sel_stim)
        inc_stim = np.where(inc[:, k])[0]
        if len(inc_stim) > 0:
            psth_inc_only[ki] = psth[inc_stim, k, :].mean(axis=0)

    has_sel = ~np.isnan(psth_selected[:, 0])
    has_inc = ~np.isnan(psth_inc_only[:, 0])
    dipper  = has_inc & ~has_sel

    fig, ax = plt.subplots(figsize=(8, 4))
    for ki in range(n_units):
        if has_sel[ki]:
            ax.plot(t_lyr, psth_selected[ki], color='steelblue', lw=0.7, alpha=0.4)
        elif dipper[ki]:
            ax.plot(t_lyr, psth_inc_only[ki], color='tomato', lw=0.7, alpha=0.5)
        else:
            ax.plot(t_lyr, psth[:, lyr_idxs[ki], :].mean(axis=0), color='grey', lw=0.4, alpha=0.15)
    if has_sel.any():
        ax.plot(t_lyr, psth_selected[has_sel].mean(axis=0), color='steelblue', lw=2.0,
                label=f'unimodal mean  (n={has_sel.sum()}, selected stim only)')
    if dipper.any():
        ax.plot(t_lyr, psth_inc_only[dipper].mean(axis=0), color='tomato', lw=2.0,
                label=f'peak-pass / unimodal-fail  (n={dipper.sum()})')
    ax.axvline(T_STIM_ON,  color='tab:blue',   lw=0.9, ls='--', label='stim onset')
    ax.axvline(T_STIM_OFF, color='tab:orange', lw=0.9, ls='--', label='stim offset')
    ax.axvspan(CWIN_LO, CWIN_HI, color='steelblue', alpha=0.07)
    ax.set_xlabel('Time from stim onset (ms)')
    ax.set_ylabel('Firing rate (Hz)')
    ax.set_title(f'Session {sd["session_id"]} — {TARGET_LAYER} ({n_units} units)\n'
                 f'Each unit averaged over its own selected stimuli only')
    ax.legend(frameon=False, fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'unit_traces_{sd["session_id"]}_{TARGET_LAYER.replace("/","_")}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    stim_win   = (t_lyr >= T_STIM_ON) & (t_lyr <= T_STIM_OFF)
    t_stim_arr = t_lyr[stim_win]
    mat_sel    = psth_selected[has_sel][:, stim_win]
    peak_bins  = np.argmax(mat_sel, axis=1)
    sort_idx   = np.argsort(peak_bins)

    fig, axes = plt.subplots(1, 2, figsize=(11, max(3, has_sel.sum() * 0.18 + 1)),
                              gridspec_kw={'width_ratios': [3, 1]})
    im = axes[0].imshow(mat_sel[sort_idx], aspect='auto', origin='upper',
                        extent=[t_stim_arr[0], t_stim_arr[-1], has_sel.sum(), 0],
                        cmap='inferno', interpolation='nearest')
    axes[0].axvline(CWIN_LO, color='cyan', lw=0.8, ls='--')
    axes[0].axvline(CWIN_HI, color='cyan', lw=0.8, ls='--')
    axes[0].set_xlabel('Time from stim onset (ms)')
    axes[0].set_ylabel('Unit (sorted by peak time)')
    axes[0].set_title(f'PSTH heatmap — {TARGET_LAYER}')
    fig.colorbar(im, ax=axes[0], label='Hz', shrink=0.8)
    axes[1].hist(t_stim_arr[peak_bins], bins=15, color='steelblue', alpha=0.8, orientation='horizontal')
    axes[1].set_xlabel('Count')
    axes[1].set_ylabel('Peak time (ms)')
    axes[1].set_title('Peak time dist.')
    axes[1].spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'unit_heatmap_{sd["session_id"]}_{TARGET_LAYER.replace("/","_")}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Units: {n_units} total  |  unimodal-contributing: {has_sel.sum()}  '
          f'|  dipper: {dipper.sum()}  |  never increasing: {(~has_inc).sum()}')
    print(f'Mean stim contributing per unit: {n_stim_sel[has_sel].mean():.1f} / {psth.shape[0]}')

### Per-session diagnostics

In [ ]:
sess_ids       = [s['session_id']              for s in per_session]
n_stim_total   = np.array([s['n_stim_total']            for s in per_session])
n_stim_contrib = np.array([s['n_stim_contributing']     for s in per_session])
mean_n_inc     = np.array([s['mean_n_increasing_units'] for s in per_session])
frac_contrib   = n_stim_contrib / n_stim_total
x      = np.arange(len(sess_ids))
labels = [str(sid)[-6:] for sid in sess_ids]
fig, axes = plt.subplots(2, 1, figsize=(max(8, len(sess_ids) * 0.6), 5), sharex=True)
axes[0].bar(x, frac_contrib * 100, color='steelblue', edgecolor='k', lw=0.4)
axes[0].axhline(100, color='grey', lw=0.8, ls='--')
axes[0].set_ylabel('% stim contributing')
axes[0].set_title('Per-session: stimulus coverage and mean increasing unit count')
axes[0].set_ylim(0, 110)
axes[0].spines[['top', 'right']].set_visible(False)
axes[1].bar(x, mean_n_inc, color='steelblue', edgecolor='k', lw=0.4)
axes[1].set_ylabel('Mean # increasing units / stim')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
axes[1].set_xlabel('Session ID (last 6 digits)')
axes[1].spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / 'per_session_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Mean % stim contributing : {frac_contrib.mean()*100:.1f}%')
print(f'Mean # increasing units  : {mean_n_inc.mean():.1f}')

### Stim-window mean vs baseline — scatter across sessions

In [ ]:
baseline_mask = (t_ms >= BL_LO)   & (t_ms < BL_HI)
stim_win_mask = (t_ms >= CWIN_LO) & (t_ms <= CWIN_HI)
sess_bl   = np.array([s['mean_hz'][baseline_mask].mean() for s in per_session])
sess_stim = np.array([s['mean_hz'][stim_win_mask].mean() for s in per_session])
ga_bl     = mu[baseline_mask].mean()
ga_stim   = mu[stim_win_mask].mean()
lim = max(sess_bl.max(), sess_stim.max()) * 1.1
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(sess_bl, sess_stim, color='steelblue', s=40, zorder=3, label='individual sessions')
ax.scatter([ga_bl], [ga_stim], color='k', s=80, marker='*', zorder=4, label='grand average')
ax.plot([0, lim], [0, lim], 'k--', lw=0.8, label='unity')
ax.set_xlabel(f'Baseline mean FR  ({BL_LO:.0f}–{BL_HI:.0f} ms)  [Hz]')
ax.set_ylabel(f'Stim-window mean FR  ({CWIN_LO:.0f}–{CWIN_HI:.0f} ms)  [Hz]')
ax.set_title('Stim response vs baseline — increasing-unit averages')
ax.legend(frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / 'stim_vs_baseline_scatter.png', dpi=150, bbox_inches='tight')
plt.show()